In [1]:
from glob import glob
import numpy as np
import pandas as pd
from pathlib import Path
# files = glob("/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_*_T1/*_1_2mW_IPhone.xls")
files = glob("/home/qivy00li/projects/gait_ml/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_*_T1/*_1_2mW_IPhone.xls")
files = np.sort(files)

files_df = pd.DataFrame(files, columns=["fname"])
files_df.head()

idx_ = files_df.fname.apply(lambda x: Path(x).stem.split("_")[0]).astype(int).values

In [2]:
import pandas as pd
import numpy as np

# # 1. Create Sample Data
# # In a real scenario, you would load your data, e.g., df = pd.read_csv('your_data.file')
# np.random.seed(42)
# n_control = 50
# n_patient = 45
# data = {
#     'group': ['Control'] * n_control + ['Patient'] * n_patient,
#     'gender': np.random.choice(['Male', 'Female'], n_control + n_patient, p=[0.55, 0.45]),
#     'age': np.concatenate([
#         np.random.normal(65, 8, n_control),
#         np.random.normal(70, 10, n_patient)
#     ]),
#     'height': np.concatenate([
#         np.random.normal(170, 10, n_control),
#         np.random.normal(168, 8, n_patient)
#     ]),
#     'weight': np.concatenate([
#         np.random.normal(75, 12, n_control),
#         np.random.normal(80, 15, n_patient)
#     ])
# }
# df = pd.DataFrame(data)
# df['age'] = df['age'].astype(int)
# df['height'] = df['height'].astype(int)
# df['weight'] = df['weight'].astype(int)

# print("--- Sample Data Head ---")
# print(df.head())
# print("\n")

df = pd.read_excel("/home/qivy00li/projects/gait_ml/data/dataset2/10092025/misc/SI_Fragebögen_bis_35.xlsx", header=4, usecols="R:X")
df = df.iloc[1:, [0,1,2,3,6]]
df.columns = ["gender", "age", "height", "weight", "group"]
df.gender.replace(0, "Male", inplace=True)
df.gender.replace(1, "Female", inplace=True)
df = df.loc[idx_]
df.dropna(inplace=True)
print("dataframe shape:", df.shape)
# df.head()


# 2. Define a function to summarize statistics
def summarize_group(data):
    """Calculates formatted stats for a given dataframe."""
    n = len(data)
    
    # Continuous variables: Mean ± SD
    age_mean_sd = f"{data['age'].mean():.1f} ± {data['age'].std():.1f}"
    height_mean_sd = f"{data['height'].mean():.1f} ± {data['height'].std():.1f}"
    weight_mean_sd = f"{data['weight'].mean():.1f} ± {data['weight'].std():.1f}"
    
    # Categorical variables: n (%)
    gender_counts = data['gender'].value_counts()
    gender_perc = data['gender'].value_counts(normalize=True) * 100
    
    # Initialize gender stats to avoid errors if a group has 0 of one gender
    gender_male_stat = "0 (0.0%)"
    gender_female_stat = "0 (0.0%)"
    
    if 'Male' in gender_counts.index:
        gender_male_stat = f"{gender_counts['Male']} ({gender_perc['Male']:.1f}%)"
    if 'Female' in gender_counts.index:
        gender_female_stat = f"{gender_counts['Female']} ({gender_perc['Female']:.1f}%)"
        
    # Create a Series (which will become a column in the final table)
    stats = pd.Series({
        'Age (years)': age_mean_sd,
        'Height (cm)': height_mean_sd,
        'Weight (kg)': weight_mean_sd,
        'Gender - Male (n, %)': gender_male_stat,
        'Gender - Female (n, %)': gender_female_stat
    })
    return stats

# 3. Build the publication table
# This will be our final table, indexed by the characteristic
publication_table = pd.DataFrame()

# Get group names
groups = df['group'].unique()
groups.sort() # Sort to ensure consistent order, e.g., 'Control' then 'Patient'

# --- Summarize each group ---
for group_name in groups:
    group_data = df[df['group'] == group_name]
    n = len(group_data)
    publication_table[f'{group_name} (n={n})'] = summarize_group(group_data)

# --- Summarize overall ---
n_total = len(df)
publication_table[f'Overall (n={n_total})'] = summarize_group(df)

# Re-order columns to have 'Overall' at the end (common practice)
cols = [col for col in publication_table.columns if col != f'Overall (n={n_total})']
cols.append(f'Overall (n={n_total})')
publication_table = publication_table[cols]

# 4. Print the final table
print("--- Publication-Ready Demographics Table ---")
print(publication_table)

# 5. Save the table to a CSV file
csv_filename = 'publication_demographics_table.csv'
# publication_table.to_csv(csv_filename)
# print(f"\nTable saved to '{csv_filename}'")

dataframe shape: (46, 5)
--- Publication-Ready Demographics Table ---
                            h (n=28)     p (n=18) Overall (n=46)
Age (years)               69.5 ± 7.0   68.8 ± 6.2     69.2 ± 6.7
Height (cm)             160.0 ± 23.1  164.8 ± 7.9   161.9 ± 18.7
Weight (kg)              74.3 ± 21.2  74.3 ± 13.5    74.3 ± 18.4
Gender - Male (n, %)       6 (21.4%)    4 (22.2%)     10 (21.7%)
Gender - Female (n, %)    22 (78.6%)   14 (77.8%)     36 (78.3%)


/tmp/ipykernel_3522814/2842408387.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.gender.replace(0, "Male", inplace=True)


In [6]:
publication_table.to_latex()

'\\begin{tabular}{llll}\n\\toprule\n & h (n=28) & p (n=18) & Overall (n=46) \\\\\n\\midrule\nAge (years) & 69.5 ± 7.0 & 68.8 ± 6.2 & 69.2 ± 6.7 \\\\\nHeight (cm) & 160.0 ± 23.1 & 164.8 ± 7.9 & 161.9 ± 18.7 \\\\\nWeight (kg) & 74.3 ± 21.2 & 74.3 ± 13.5 & 74.3 ± 18.4 \\\\\nGender - Male (n, %) & 6 (21.4%) & 4 (22.2%) & 10 (21.7%) \\\\\nGender - Female (n, %) & 22 (78.6%) & 14 (77.8%) & 36 (78.3%) \\\\\n\\bottomrule\n\\end{tabular}\n'

In [4]:
!python -m pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]━━ 1/2 [openpyxl]
